# Модуль 5. Работа с несбалансированными данными: ресемплинг и ловушки

**Длительность:** 120 минут
**Формат:** теория (85 мин) + практика (35 мин)
**Цель модуля:** Учащийся должен понимать не только «как включить `class_weight` или SMOTE», но и **что именно происходит математически** при их применении: как `class_weight` искажает выходные вероятности модели через изменение эффективных априорных вероятностей классов, и почему SMOTE, примененный до кросс-валидации, создает утечку данных. По итогам модуля учащийся должен уметь построить корректный CV-пайплайн с ресемплингом и объяснить на формулах, почему «грязная» версия того же пайплайна дает завышенные метрики.

## 1. Введение: два разных способа бороться с дисбалансом (5 мин)

### 1.1. Что мы уже умеем
В Модуле 4 мы научились работать с дисбалансом «на выходе» модели: обучаем модель как есть, получаем сырые вероятности, затем подбираем порог, минимизирующий бизнес-издержки. Этот подход ничего не меняет в процессе обучения — вся работа происходит постфактум, над готовыми предсказаниями.

### 1.2. Альтернативный подход: менять сам процесс обучения

Есть другая философия: заставить **саму модель** во время обучения уделять больше внимания редкому классу. Два основных инструмента:
- **`class_weight`** — не меняет данные, а меняет функцию потерь: ошибка на объекте редкого класса «весит» больше при расчете градиента.
- **SMOTE** (Synthetic Minority Over-sampling Technique) — не меняет функцию потерь, а меняет сами данные: генерирует синтетические объекты редкого класса, физически увеличивая его долю в обучающей выборке.

### 1.3. Почему это не просто «еще один инструмент», а источник новых рисков

Оба инструмента решают проблему дисбаланса, но создают два новых риска, каждый из которых при невнимательном обращении полностью обесценивает всю проделанную работу:
1. **`class_weight` систематически искажает вероятности** — модель после такого обучения выдает не честные $P(y=1\mid x)$, а вероятности, «сдвинутые» в сторону редкого класса. Если бэкенду нужны настоящие вероятности для финансового расчета (как в Модуле 4), это критично.
2. **SMOTE, примененный до разбиения на фолды, создает утечку данных** — эффект, тонкий по механизму, но катастрофический по последствиям: метрики на валидации оказываются завышены, и после выкладки в продакшен модель «внезапно» работает намного хуже, чем показывала кросс-валидация.

Этот модуль последовательно разбирает механизм обеих проблем и дает работающую архитектуру, которая их избегает.

## 2. `class_weight`: механизм изменения весов в функции потерь (15 мин)

### 2.1. Обычная (невзвешенная) функция потерь
Как обычно вычисляется LogLoss для бинарной классификации:
$$L = -\frac{1}{N}\sum_{i=1}^N \left[y_i \log(p_i) + (1-y_i)\log(1-p_i)\right]$$
Каждый объект вносит одинаковый вклад в общую сумму, независимо от того, к какому классу он принадлежит.

### 2.2. Взвешенная функция потерь
`class_weight` вводит множитель $w_{y_i}$, зависящий от класса объекта:
$$L_w = -\frac{1}{N}\sum_{i=1}^N w_{y_i}\left[y_i \log(p_i) + (1-y_i)\log(1-p_i)\right]$$
где $w_{y_i} = w_1$, если $y_i = 1$, и $w_{y_i} = w_0$, если $y_i = 0$. Ошибка на объекте с большим весом дает больший вклад в градиент — оптимизатор «сильнее» реагирует на ошибки этого класса.

### 2.3. Формула `class_weight='balanced'` в Scikit-Learn
$$w_c = \frac{N}{K \cdot N_c}$$
где $N$ — общее число объектов, $K$ — число классов, $N_c$ — число объектов класса $c$. Для бинарной классификации ($K=2$):
$$w_1 = \frac{N}{2 N_1}, \qquad w_0 = \frac{N}{2 N_0}$$

### 2.4. Ключевое свойство: выравнивание суммарного веса классов
Проверим, что дают эти веса, если умножить их на число объектов соответствующего класса:
$$w_1 \cdot N_1 = \frac{N}{2N_1}\cdot N_1 = \frac{N}{2}, \qquad w_0 \cdot N_0 = \frac{N}{2N_0}\cdot N_0 = \frac{N}{2}$$
**Оба произведения равны $N/2$ — то есть суммарный «вес», который каждый класс вносит в функцию потерь, становится одинаковым, независимо от того, насколько исходно различалось число объектов в классах.** Это и есть формальный смысл слова «balanced».

### 2.5. Числовой пример
$N = 10\,000$, $N_1 = 100$ (positive, 1 %), $N_0 = 9\,900$ (negative, 99 %):
$$w_1 = \frac{10\,000}{2 \times 100} = 50, \qquad w_0 = \frac{10\,000}{2 \times 9\,900} \approx 0.505$$
$$\frac{w_1}{w_0} = \frac{N_0}{N_1} = \frac{9\,900}{100} = 99$$
Ошибка на одном-единственном positive объекте весит в функции потерь **в 99 раз больше**, чем ошибка на одном negative объекте — ровно во столько раз, во сколько positive класс малочисленнее.

### 2.6. LightGBM: те же идеи, другие названия параметров

В LightGBM аналог `class_weight='balanced'` реализован двумя способами:
- `is_unbalance=True` — библиотека сама вычисляет соотношение классов и взвешивает функцию потерь аналогично формуле из раздела 2.3;
- `scale_pos_weight=<число>` — ручное задание отношения $N_0/N_1$ (или любого другого желаемого соотношения весов positive к negative).

Эти два параметра нельзя использовать одновременно — они управляют одним и тем же механизмом.

## 3. Математический эффект: искусственное изменение априорных вероятностей классов (20 мин)

Это центральный по важности раздел модуля: он объясняет **почему** взвешивание влияет не только на то, как обучается модель, но и на то, **что именно означают её выходные вероятности**.

### 3.1. Байесовское разложение истинной вероятности класса
Пусть $\pi_1 = P(y=1)$ и $\pi_0 = P(y=0) = 1-\pi_1$ — истинные априорные вероятности классов в генеральной совокупности (то есть реальная доля positive объектов в природе, а не в конкретной выборке). Пусть $P(x\mid y=1)$ и $P(x\mid y=0)$ — распределения признаков внутри каждого класса (эти распределения — объективное свойство данных, они не меняются от того, как мы взвешиваем функцию потерь). По формуле Байеса истинная апостериорная вероятность:
$$P(y=1\mid x) = \frac{\pi_1\, P(x\mid y=1)}{\pi_1\, P(x\mid y=1) + \pi_0\, P(x\mid y=0)}$$

### 3.2. Что делает взвешенная функция потерь на уровне математического ожидания
Рассмотрим ожидаемое значение взвешенного LogLoss (переходя от суммы по конечной выборке к математическому ожиданию по генеральной совокупности — это законно в пределе большого $N$):
$$E_{(x,y)\sim P}\big[w_y \cdot \ell(y, f(x))\big] = \pi_1 w_1 \cdot E_{x\sim P(x\mid y=1)}\big[\ell(1,f(x))\big] \;+\; \pi_0 w_0 \cdot E_{x\sim P(x\mid y=0)}\big[\ell(0,f(x))\big]$$

Введем нормировочную константу $Z = \pi_1 w_1 + \pi_0 w_0$ и перепишем выражение:
$$= Z \cdot \left[\underbrace{\frac{\pi_1 w_1}{Z}}_{\pi_1'} \cdot E_{x\sim P(x\mid y=1)}[\ell(1,f(x))] + \underbrace{\frac{\pi_0 w_0}{Z}}_{\pi_0'}\cdot E_{x\sim P(x\mid y=0)}[\ell(0,f(x))]\right]$$

Поскольку $Z$ — положительная константа, не зависящая от $f$, минимизация выражения слева по $f$ дает **тот же минимизатор**, что и минимизация выражения в квадратных скобках. А выражение в скобках — это в точности **обычный (невзвешенный) ожидаемый LogLoss**, но вычисленный при **других** априорных вероятностях классов $\pi_1' = \pi_1 w_1/Z$ и $\pi_0' = \pi_0 w_0/Z$ вместо истинных $\pi_1, \pi_0$.

### 3.3. Следствие: модель обучается под «искусственный мир»
LogLoss — строго правильное правило оценивания (proper scoring rule): его минимизатор по $f$ — это именно апостериорная вероятность при тех приорах, что фактически «зашиты» в ожидание. Значит, при достаточном объеме данных и достаточной емкости модели, взвешенное обучение сходится не к истинной $P(y=1\mid x)$, а к:
$$f^*(x) = P'(y=1\mid x) = \frac{\pi_1' P(x\mid y=1)}{\pi_1' P(x\mid y=1) + \pi_0' P(x\mid y=0)}$$
— апостериорной вероятности **под искусственными приорами** $\pi_1', \pi_0'$, а не под реальными $\pi_1, \pi_0$.

### 3.4. Что дает конкретно `'balanced'`
Подставим формулы весов из раздела 2.3, выразив их через приоры ($w_1 = N/(2N_1) = 1/(2\pi_1)$, $w_0 = 1/(2\pi_0)$, так как $N_1/N=\pi_1$):
$$\pi_1 w_1 = \pi_1 \cdot \frac{1}{2\pi_1} = \frac{1}{2}, \qquad \pi_0 w_0 = \pi_0 \cdot \frac{1}{2\pi_0} = \frac{1}{2}$$
$$Z = \frac12+\frac12 = 1 \quad\Rightarrow\quad \pi_1' = \frac{1/2}{1} = 0.5, \qquad \pi_0' = 0.5$$

**Вывод: `class_weight='balanced'` заставляет модель обучаться так, будто в природе классы распределены строго 50/50 — независимо от того, насколько сильно они несбалансированы на самом деле.** Это математически точное подтверждение интуитивного описания из технической документации («веса обратно пропорциональны частоте класса») — здесь показано, к чему именно это приводит на уровне выходной вероятности, а не только на уровне градиента.

### 3.5. Важная оговорка о строгости результата
Строгий вывод раздела 3.2–3.3 точен для LogLoss и моделей, оптимизирующих его напрямую (логистическая регрессия, нейросети с сигмоидным выходом). Для древесных ансамблей (LightGBM, XGBoost) внутренняя механика взвешивания устроена немного иначе технически (веса влияют на расчет градиентов и гессианов при построении каждого дерева), но итоговый качественный эффект — систематический сдвиг выходных вероятностей в сторону редкого класса — сохраняется и подтверждается эмпирически (раздел 9, практика). Разница в строгости вывода между линейными моделями и бустингом — уместный аргумент на собеседовании, если попросят обосновать заявление «class_weight искажает вероятности» не только качественно, но и математически.

## 4. Probability Shifting: практическое проявление и формула коррекции (15 мин)

### 4.1. Что видно на практике
Раздел 3 показал: модель с `class_weight='balanced'` обучается под приор 50/50. На практике это означает, что модель начинает выдавать **завышенные** вероятности для редкого (positive) класса: объекты, у которых в реальности шанс быть positive составляет условно 2 %, модель после взвешенного обучения может оценивать в 30–40 % — потому что она «не знает», что в реальном мире positive класс редок, она обучена так, будто он встречается так же часто, как negative.

### 4.2. Почему это критично для бэкенда
Вспомним Модуль 4: поиск оптимального порога и любые финансовые расчеты (`FP(t)\cdot C_{FP} + FN(t)\cdot C_{FN}`) неявно опираются на то, что вероятность $p$ отражает реальный шанс события. Если вместо честных $P(y=1\mid x)$ модель отдает искусственно завышенные $P'(y=1\mid x)$, вся логика подбора порога по разделу 5 предыдущего модуля начинает работать над неверными числами — оптимальный порог, найденный сканированием, будет верным *для искаженной шкалы вероятностей*, но использовать сырое значение $p'$ как «вероятность фрода» в отчете для бизнеса или в другой части системы, ожидающей честную вероятность, — ошибка.

### 4.3. Вывод формулы коррекции (алгебраическое «обращение» сдвига приоров)
Из формулы Байеса (раздел 3.1) можно выразить **отношение шансов** (odds) через приоры и отношение правдоподобий, которое, важно отметить, **не зависит от того, как мы взвешиваем обучение** — оно является объективным свойством данных $P(x\mid y=1)/P(x\mid y=0)$:
$$\text{odds}_{\text{true}}(x) = \frac{P(y=1\mid x)}{P(y=0\mid x)} = \frac{\pi_1}{\pi_0}\cdot\frac{P(x\mid y=1)}{P(x\mid y=0)}$$
$$\text{odds}_{\text{model}}(x) = \frac{P'(y=1\mid x)}{P'(y=0\mid x)} = \frac{\pi_1'}{\pi_0'}\cdot\frac{P(x\mid y=1)}{P(x\mid y=0)}$$

Разделив первое на второе, отношение правдоподобий $P(x\mid y=1)/P(x\mid y=0)$ сокращается:
$$\text{odds}_{\text{true}}(x) = \underbrace{\frac{\pi_1/\pi_0}{\pi_1'/\pi_0'}}_{r} \cdot \text{odds}_{\text{model}}(x)$$

Для случая `'balanced'` ($\pi_1'=\pi_0'=0.5$, то есть $\pi_1'/\pi_0'=1$) коэффициент упрощается до $r = \pi_1/\pi_0$ — обычного истинного отношения шансов классов в природе.

### 4.4. Итоговая формула
Обозначим сырой (искаженный) выход модели как $p' = P'(y=1\mid x)$. Тогда скорректированная («честная») вероятность:
$$p_{\text{corrected}} = \frac{r \cdot \dfrac{p'}{1-p'}}{1 + r\cdot\dfrac{p'}{1-p'}}, \qquad r = \frac{\pi_1}{\pi_0}$$

### 4.5. Проверка на здравый смысл
Пусть истинная доля positive $\pi_1 = 0.01$ ($\pi_0=0.99$, $r\approx 0.0101$), и модель выдала $p' = 0.5$ («после балансировки модель полностью не уверена»). Тогда $\text{odds}_{\text{model}} = 1$, $\text{odds}_{\text{true}} = r \cdot 1 \approx 0.0101$, а $p_{\text{corrected}} = 0.0101/1.0101 \approx 0.01$.

Результат в точности совпадает с базовой долей positive класса в природе ($\pi_1=0.01$) — что логично: если модель, обученная под искусственный приор 50/50, «пожимает плечами» и говорит 50/50 по конкретному объекту, значит, для этого объекта у неё нет никакого дополнительного сигнала сверх базовой частоты события, и при переводе обратно в реальный мир мы должны получить именно базовую частоту, а не 50 %. Формула прошла проверку.

### 4.6. Практическая оговорка
Эта коррекция — точная только при условиях раздела 3.5 (LogLoss, достаточно данных, приблизительно верно и для приближенно откалиброванных моделей). На практике эмпирическая калибровка (Platt Scaling, Isotonic Regression — Модуль 8) — более надежный и менее чувствительный к допущениям способ вернуть честные вероятности, потому что она не полагается на точное совпадение теоретической модели взвешивания с тем, что реально происходит внутри конкретной реализации алгоритма (особенно для бустинга, см. раздел 3.5). Формула этого раздела ценна прежде всего для понимания **природы** искажения, а не как рекомендуемый production-инструмент.

### 4.7. Численная проверка (код)

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# 1. Датасет с известным истинным дисбалансом
X, y = make_classification(
    n_samples=50_000, n_features=10, n_informative=6,
    weights=[0.95, 0.05], flip_y=0.01, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

pi_1 = y_train.mean()   # истинная доля positive в обучающей выборке
pi_0 = 1 - pi_1

# 2. Модель БЕЗ взвешивания — приближение "честных" вероятностей
model_true = LogisticRegression(max_iter=1000, random_state=42)
model_true.fit(X_train, y_train)
p_true_approx = model_true.predict_proba(X_test)[:, 1]

# 3. Модель С class_weight='balanced' — искаженные вероятности
model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model_balanced.fit(X_train, y_train)
p_prime = model_balanced.predict_proba(X_test)[:, 1]

# 4. Коррекция по формуле раздела 4.4 (r = pi_1/pi_0, т.к. balanced -> pi_1'=pi_0'=0.5)
odds_prime = p_prime / (1 - p_prime)
r = pi_1 / pi_0
true_odds = r * odds_prime
p_corrected = true_odds / (1 + true_odds)

# 5. Сравнение
print(f"Средняя p' (balanced, ДО коррекции):        {p_prime.mean():.4f}")
print(f"Средняя p_corrected (ПОСЛЕ коррекции):       {p_corrected.mean():.4f}")
print(f"Средняя p (без взвешивания, эталон):         {p_true_approx.mean():.4f}")
print(f"Средняя |p_prime - эталон| (без коррекции):  {np.abs(p_prime - p_true_approx).mean():.4f}")
print(f"Средняя |p_corrected - эталон| (с коррекцией): {np.abs(p_corrected - p_true_approx).mean():.4f}")

Ожидаемый результат: `p_prime` в среднем заметно выше `p_true_approx` (искажение подтверждено), а `p_corrected` заметно ближе к `p_true_approx`, чем `p_prime` — формула коррекции работает и эмпирически, не только теоретически.

## 5. SMOTE: генерация синтетических точек (15 мин)

### 5.1. Идея
SMOTE (Synthetic Minority Over-sampling Technique, Chawla et al., 2002) не переиспользует существующие объекты редкого класса напрямую (в отличие от простого дублирования — Random Oversampling), а **создает новые, синтетические** объекты редкого класса путем интерполяции между реальными.

### 5.2. Алгоритм пошагово
1. Для каждого объекта $x_i$ редкого (минорного) класса найти его $k$ ближайших соседей **среди объектов того же минорного класса** (обычно $k=5$, метрика — евклидово расстояние в пространстве признаков).
2. Случайно выбрать одного соседа $x_{nn}$ из этих $k$.
3. Сгенерировать синтетическую точку на отрезке, соединяющем $x_i$ и $x_{nn}$:
$$x_{new} = x_i + \lambda \cdot (x_{nn} - x_i), \qquad \lambda \sim \text{Uniform}(0,1)$$
Метка новой точки — тот же минорный класс.
4. Повторять шаги 1–3, пока не будет сгенерировано нужное число синтетических точек (обычно — пока минорный класс не достигнет желаемого соотношения с мажоритарным, по умолчанию — паритета 1:1).

### 5.3. Числовой пример на плоскости
Пусть $x_i = (2, 3)$ — минорный объект, его ближайший сосед того же класса $x_{nn} = (4, 7)$. Случайно выбрано $\lambda = 0.3$:
$$x_{new} = (2,3) + 0.3 \cdot \big((4,7)-(2,3)\big) = (2,3) + 0.3\cdot(2,4) = (2,3)+(0.6,\,1.2) = (2.6,\ 4.2)$$
Новая точка лежит на отрезке между $x_i$ и $x_{nn}$, ближе к $x_i$ (поскольку $\lambda=0.3 < 0.5$).

### 5.4. Ограничения метода
- **Только числовые признаки.** Линейная интерполяция между категориями не имеет смысла («интерполяция между городом Москва и городом Казань» не определена) — для смешанных данных существуют варианты `SMOTENC` (Nominal and Continuous) и `SMOTEN` (только категориальные), не входящие в объем этого модуля.
- **Предположение о локальной линейности.** Метод неявно предполагает, что пространство между двумя близкими минорными объектами тоже принадлежит минорному классу. Это разумное допущение в плотных, хорошо разделимых областях признакового пространства, но ломается вблизи границы классов или в шумных, разреженных зонах: синтетическая точка может оказаться в области, фактически принадлежащей мажоritarному классу («перетекание» синтетических точек через границу решения — известная критика метода в литературе).
- **Чувствительность к выбросам.** Если $x_i$ — выброс (аномальная точка минорного класса, физически удаленная от остальных), интерполяция с его «ближайшими соседями» (тоже удаленными) создает синтетические точки в малоинформативной, нерепрезентативной области.

## 6. Data Leakage при ресемплинге: почему SMOTE до кросс-валидации катастрофичен (15 мин)

### 6.1. Формулировка проблемы
Если SMOTE применяется **ко всему датасету целиком**, а разбиение на фолды кросс-валидации происходит **после** этого — синтетические точки, которые окажутся в обучающей части фолда, могут быть построены на основе реальных точек, которые окажутся в валидационной части того же фолда. Обучающая и валидационная выборки перестают быть по-настоящему независимыми, хотя формально это не видно ни в коде, ни в размерах массивов.

### 6.2. Механизм №1: синтетические точки — почти дубликаты валидационных объектов
Рассмотрим 5 реальных минорных объектов: $A, B, C, D, E$. SMOTE, примененный к полному датасету, создает синтетическую точку $S_1$, интерполируя между $A$ и $B$: $S_1 = A + \lambda(B-A)$.

Теперь происходит случайное разбиение на фолды кросс-валидации. Предположим, точка $A$ по случайности попадает в **валидационный** фолд, а точки $S_1$ и $B$ — в **обучающий**. При малом $\lambda$ (например, $\lambda=0.05$) точка $S_1$ находится очень близко к $A$ в признаковом пространстве — практически совпадает с ней. Модель во время обучения «видит» объект, почти неотличимый от $A$, а затем ее оценивают на самой $A$ в валидации. Модель предсказывает $A$ значительно точнее, чем предсказала бы объект, которого никогда «не видела» — валидационная метрика оказывается завышена не потому, что модель действительно хорошо обобщает, а потому, что она частично «подглядела» ответ.

### 6.3. Механизм №2: сам поиск соседей использует данные из будущей валидации
Есть более тонкий и более общий канал утечки, не требующий, чтобы синтетическая точка оказалась близкой копией конкретного валидационного объекта. Шаг 1 алгоритма SMOTE (поиск $k$ ближайших соседей) выполняется **по всему датасету**, включая те объекты, которые впоследствии окажутся в валидационном фолде. Это означает, что даже структура и плотность синтетических точек (не только их отдельные координаты) неявно формируется с учетом информации о том, где именно располагаются будущие валидационные объекты. Формально обучающая выборка данного фолда перестает быть независимой от валидационной выборки того же фолда, потому что обе выведены из одного и того же нерасщепленного облака точек до разбиения.

### 6.4. Итог: почему это «катастрофично», а не просто «неточно»
В отличие от многих источников небольшой погрешности в валидации, эта утечка систематически **завышает** метрики качества, причем тем сильнее, чем агрессивнее ресемплинг (чем больше синтетических точек создается относительно объема реальных данных). На практике это означает: команда видит впечатляющий PR-AUC на кросс-валидации, принимает решение о выкладке модели в продакшен, а затем сталкивается с резким падением реального качества — ровно тот сценарий, ради предотвращения которого был задуман весь Модуль 1 (честная валидация). Ресемплинг, примененный неаккуратно, сводит на нет всю дисциплину честного разбиения данных.

## 7. Правильная архитектура: разбиение -> SMOTE только внутри train-фолда (10 мин)

### 7.1. Общее правило
$$\text{Split} \;\rightarrow\; \text{SMOTE только на train-части каждого фолда} \;\rightarrow\; \text{Fit} \;\rightarrow\; \text{Predict на нетронутом validation-фолде}$$

Валидационный фолд **никогда** не проходит через SMOTE — он должен оставаться отражением реального, естественно несбалансированного распределения классов, потому что именно с таким распределением модель столкнется в продакшене. Оценивать модель на искусственно сбалансированной валидации — методологическая ошибка отдельного рода: PR-AUC (Модуль 3) имеет смысл именно относительно реальной prevalence, и искусственно «выровненная» валидация делает саму метрику неинтерпретируемой.

### 7.2. Почему `sklearn.pipeline.Pipeline` здесь не подходит
Стандартный `Pipeline` из Scikit-Learn построен на контракте: каждый промежуточный шаг — это `Transformer` с методом `.transform(X)`, который принимает $X$ и возвращает $X'$ **того же числа строк**, не имея доступа к $y$ вообще. SMOTE физически не вписывается в этот контракт: ему нужен доступ и к $X$, и к $y$ (чтобы знать, какие объекты — минорный класс), и он **меняет число строк** (добавляет синтетические). Попытка засунуть `SMOTE` в обычный `Pipeline` вызовет ошибку интерфейса.

### 7.3. `imblearn.pipeline.Pipeline`

Библиотека `imbalanced-learn` предоставляет собственный класс `Pipeline`, поддерживающий особый тип шагов — **sampler** (реализующий `.fit_resample(X, y) -> (X_new, y_new)`). Ключевой механизм, который делает его автоматически безопасным для CV:
- на вызове `.fit(X, y)` sampler-шаг **активен**: он ресэмплит переданные ему данные;
- на вызовах `.predict(X)` / `.transform(X)` (то есть на инференсе — в том числе внутри `cross_val_score` при оценке на валидационном фолде) sampler-шаг **пропускается** (действует как identity-преобразование) — предсказание всегда делается на исходных, нетронутых данных.

Именно поэтому `cross_val_score(imblearn_pipeline, X, y, cv=...)` автоматически, без ручного написания цикла, гарантирует: на каждом сплите `.fit()` вызывается на обучающей части фолда (где SMOTE активен), а `.predict()` — на валидационной части (где SMOTE неактивен). Это единственный инструмент из рассмотренных, который делает правильное поведение архитектурно неизбежным, а не зависящим от дисциплины разработчика.

## 8. Практика 1: корректный CV-цикл с SMOTE (35 мин)

### 8.1. Постановка задачи
Реализовать два эквивалентных способа корректной кросс-валидации с SMOTE (ручной цикл и `imblearn.pipeline.Pipeline`), затем явно продемонстрировать эффект утечки данных, сравнив с «грязной» версией, где SMOTE применен до разбиения.

### 8.2. Способ 1: ручной цикл `StratifiedKFold`

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from imblearn.over_sampling import SMOTE

# 1. Датасет с дисбалансом 1:99
X, y = make_classification(
    n_samples=20_000, n_features=20, n_informative=10,
    weights=[0.99, 0.01], flip_y=0.01, random_state=42
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pr_auc_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
    y_train_fold, y_val_fold = y[train_idx], y[val_idx]

    # SMOTE — ТОЛЬКО на обучающей части текущего фолда
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_fold, y_train_fold)

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train_res, y_train_res)

    # Валидационный фолд остаётся НЕТРОНУТЫМ — естественный дисбаланс
    y_proba_val = model.predict_proba(X_val_fold)[:, 1]
    score = average_precision_score(y_val_fold, y_proba_val)
    pr_auc_scores.append(score)

    print(f"Фолд {fold}: PR-AUC = {score:.4f}  "
          f"(train до SMOTE: {len(y_train_fold)} строк, после: {len(y_train_res)} строк)")

print(f"\nСредний PR-AUC (корректная валидация): "
      f"{np.mean(pr_auc_scores):.4f} ± {np.std(pr_auc_scores):.4f}")

### 8.3. Способ 2: `imblearn.pipeline.Pipeline` + `cross_val_score`

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import cross_val_score

imb_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_pipeline = cross_val_score(imb_pipeline, X, y, cv=skf, scoring='average_precision')

print(f"PR-AUC по фолдам (Pipeline): {np.round(scores_pipeline, 4)}")
print(f"Средний PR-AUC (Pipeline):   {scores_pipeline.mean():.4f} ± {scores_pipeline.std():.4f}")

Способ 1 и Способ 2 должны давать очень близкие (в пределах случайности `random_state`) значения среднего PR-AUC — это подтверждает, что `imblearn.pipeline.Pipeline` действительно реализует ту же логику, что и ручной цикл, просто более компактно и без риска забыть где-то применить SMOTE неправильно.

### 8.4. Демонстрация утечки: «грязная» версия для сравнения

In [ ]:
# НЕПРАВИЛЬНО: SMOTE применяется ДО разбиения на фолды
smote_global = SMOTE(random_state=42)
X_resampled_global, y_resampled_global = smote_global.fit_resample(X, y)

skf_wrong = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_wrong = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X_resampled_global, y_resampled_global,
    cv=skf_wrong, scoring='average_precision'
)

print(f"\n--- Итоговое сравнение ---")
print(f"PR-AUC, SMOTE ДО разбиения (с утечкой):      {scores_wrong.mean():.4f}")
print(f"PR-AUC, SMOTE ТОЛЬКО на train (корректно):   {scores_pipeline.mean():.4f}")
print(f"Завышение метрики из-за утечки:              {scores_wrong.mean() - scores_pipeline.mean():.4f}")

### 8.5. Что должно получиться и как это интерпретировать
`scores_wrong.mean()` должен оказаться заметно выше `scores_pipeline.mean()` — это и есть количественное, измеримое в числах подтверждение эффекта из раздела 6: «грязная» валидация систематически рисует более оптимистичную картину, чем есть на самом деле. Разница будет тем заметнее, чем сильнее дисбаланс (чем больше синтетических точек приходится генерировать относительно объема реальных данных минорного класса) — полезно проверить эту зависимость самостоятельно, изменив `weights` на более и менее экстремальные значения.

## 9. Практика 2: калибровочные кривые для `class_weight` (15 мин)

### 9.1. Постановка задачи
Обучить `LGBMClassifier` дважды — без взвешивания и с `class_weight='balanced'` — и визуально, через калибровочную кривую (Reliability Diagram), зафиксировать сдвиг вероятностей, теоретически предсказанный в разделе 3. Полноценная теория калибровки будет разобрана в Модуле 8; здесь `calibration_curve` используется как готовый диагностический инструмент.

### 9.2. Код

In [ ]:
import lightgbm as lgb
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
import matplotlib.pyplot as plt

X, y = make_classification(
    n_samples=50_000, n_features=20, n_informative=10,
    weights=[0.95, 0.05], flip_y=0.01, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Модель БЕЗ взвешивания
model_plain = lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1)
model_plain.fit(X_train, y_train)
p_plain = model_plain.predict_proba(X_test)[:, 1]

# Модель С class_weight='balanced'
model_balanced = lgb.LGBMClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, verbose=-1
)
model_balanced.fit(X_train, y_train)
p_balanced = model_balanced.predict_proba(X_test)[:, 1]

# Калибровочные кривые (quantile-бины — примерно равное число объектов в каждом бине)
frac_pos_plain, mean_pred_plain = calibration_curve(y_test, p_plain, n_bins=10, strategy='quantile')
frac_pos_balanced, mean_pred_balanced = calibration_curve(y_test, p_balanced, n_bins=10, strategy='quantile')

plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], '--', color='gray', label='Идеальная калибровка')
plt.plot(mean_pred_plain, frac_pos_plain, marker='o', label='Без class_weight')
plt.plot(mean_pred_balanced, frac_pos_balanced, marker='s', label="class_weight='balanced'")
plt.xlabel('Средняя предсказанная вероятность в бине')
plt.ylabel('Реальная доля positive-объектов в бине')
plt.title('Reliability Diagram: влияние class_weight на калибровку LightGBM')
plt.legend()
plt.grid(True)
plt.show()

print(f"Brier score, без class_weight:         {brier_score_loss(y_test, p_plain):.5f}")
print(f"Brier score, class_weight='balanced':  {brier_score_loss(y_test, p_balanced):.5f}")

### 9.3. Как читать результат
- Кривая для модели **без** `class_weight` должна проходить ближе к диагонали (идеальной калибровке).
- Кривая для модели **с** `class_weight='balanced'` должна систематически лежать **ниже** диагонали в средней и правой части графика: это означает, что при заданной предсказанной вероятности (например, 0.5) реальная доля positive среди таких объектов на самом деле заметно ниже (например, 0.15) — то есть модель завышает вероятность, ровно как предсказывает раздел 3–4.
- Brier score раскладывается на компоненту калибровки и компоненту различающей способности (refinement); `class_weight` обычно ухудшает именно калибровочную компоненту, поэтому итоговый Brier score чаще (но не гарантированно для любых данных) оказывается хуже у взвешенной модели — само сравнение чисел в конкретном запуске стоит воспринимать как иллюстрацию, а не как универсальный закон.

### 9.4. Связь с проектом FraudGuard
В коде обучения FraudGuard (`train.py`) используется ровно `class_weight='balanced'` для LightGBM-бейзлайна. Это означает, что сырые `predict_proba` этой модели подвержены именно тому искажению, которое разобрано в этом модуле — при построении Cost-Sensitive Threshold (Модуль 4 плана, уже реализованный в проекте на Дне 3) это не является ошибкой, поскольку сканирование порогов работает с сырыми вероятностями «как есть» и не требует их калиброванности. Но если бы потребовалось выводить `fraud_probability` наружу как самостоятельную интерпретируемую величину (например, в отчете для комплаенс-отдела банка), эти вероятности потребовали бы предварительной калибровки (Модуль 8).

## 10. Итоги модуля (5 мин)

### Ключевые тезисы
1. `class_weight='balanced'` вычисляется как $w_c = N/(K\cdot N_c)$ и обладает свойством: суммарный взвешенный вклад каждого класса в функцию потерь становится равным $N/2$, независимо от исходного дисбаланса.
2. Математически взвешивание LogLoss эквивалентно обучению под искусственным приором классов; для `'balanced'` этот искусственный приор всегда 50/50, независимо от реального соотношения классов.
3. Следствие — Probability Shifting: выходные вероятности взвешенной модели систематически завышены для редкого класса; формула $p_{\text{corrected}} = \dfrac{r\cdot p'/(1-p')}{1+r\cdot p'/(1-p')}$ с $r=\pi_1/\pi_0$ позволяет теоретически восстановить честную вероятность, но на практике эмпирическая калибровка (Модуль 8) надежнее.
4. SMOTE создает синтетические объекты линейной интерполяцией между реальным минорным объектом и его ближайшим соседом того же класса: $x_{new}=x_i+\lambda(x_{nn}-x_i)$.
5. Применение SMOTE до разбиения на фолды создает утечку данных двумя механизмами: синтетические точки могут быть «почти дубликатами» будущих валидационных объектов, а сам поиск соседей использует информацию из будущей валидационной выборки.
6. Единственная надежная архитектура: разбиение на фолды -> SMOTE только внутри обучающей части каждого фолда -> валидация на нетронутых естественно несбалансированных данных. `imblearn.pipeline.Pipeline` реализует это автоматически, так как sampler-шаги активны только на `.fit()` и пропускаются на `.predict()`/`.transform()`.

### Контрольные вопросы
- Выведите формулы весов `class_weight='balanced'` и покажите, что суммарный вес каждого класса в функции потерь становится равным $N/2$.
- Почему минимизация взвешенного LogLoss эквивалентна минимизации обычного LogLoss под другим набором априорных вероятностей классов? На каком свойстве LogLoss как правильного правила оценивания это основано?
- Опишите оба механизма утечки данных при применении SMOTE до кросс-валидации — почему одного упоминания «синтетические точки похожи на реальные» недостаточно для полного объяснения?
- Почему `sklearn.pipeline.Pipeline` технически не может содержать шаг SMOTE, а `imblearn.pipeline.Pipeline` может?
- Если бы вместо `class_weight='balanced'` для устранения дисбаланса использовался SMOTE, ожидали бы вы того же эффекта Probability Shifting? Обоснуйте на уровне механизма (не только «да/нет»).

### Что дальше
В следующем модуле мы разберем более общий источник завышенных метрик на валидации — Data Leakage через препроцессинг (масштабирование, заполнение пропусков, кодирование категорий) — и единственный надежный способ защиты от него: инкапсуляцию всех шагов препроцессинга в `Pipeline` с `ColumnTransformer`, обобщающий идею, уже примененную сегодня к SMOTE.